# Posterior HDF5 recover test

Discover/filter `(decoded_posteriors)_tbin-*.h5` like the paper archive notebook, load arrays via `LoadedPosteriorContainer`, then recover original `SingleEpochDecodedResult` objects with `read_hdf`.


In [ ]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3

# required for PhoPaginatedMultiDecoderDecodedEpochsWindow / Qt widgets:
%gui qt5

from copy import deepcopy
from typing import Dict, List, Tuple, Optional, Any
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import h5py

pd.set_option('display.max_columns', 100)

from neuropy.core.user_annotations import UserAnnotationsManager
from neuropy.utils.result_context import IdentifyingContext
from pyphocorehelpers.Filesystem.path_helpers import find_first_extant_path
from pyphocorehelpers.Filesystem.HDF5.hdf5_file_helpers import HDF5_Helper
from pyphoplacecellanalysis.SpecificResults.AcrossSessionResults import load_across_sessions_exported_h5_files, get_only_most_recent_session_files, find_csv_files, find_most_recent_files, AcrossSessionHelpers
from pyphoplacecellanalysis.Pho2D.data_exporting import PosteriorExporting, LoadedPosteriorContainer
from pyphoplacecellanalysis.Analysis.Decoder.reconstruction import SingleEpochDecodedResult, DecodedFilterEpochsResult
from pyphoplacecellanalysis.General.Pipeline.Stages.DisplayFunctions.DecoderPredictionError import OverlayLabelsPlotData
from neuropy.core.epoch import ensure_dataframe
from pyphocorehelpers.assertion_helpers import Assert
from pyphocorehelpers.print_helpers import print_keys_if_possible

## Resolve `collected_outputs_directory`


In [ ]:
known_collected_outputs_paths = [Path(v).resolve() for v in ['/Users/pho/data/collected_outputs',
                                                            '/Volumes/SwapSSD/Data/collected_outputs', r"K:/scratch/collected_outputs", '/Users/pho/Dropbox (University of Michigan)/MED-DibaLabDropbox/Data/Pho/Outputs/output/collected_outputs', r'C:/Users/pho/repos/Spike3DWorkEnv/Spike3D/output/collected_outputs',
                                                            '/home/halechr/cloud/turbo/Data/Output/collected_outputs',
                                                            '/home/halechr/cloud/turbo/Pho/Output/collected_outputs',
                                                            '/home/halechr/FastData/collected_outputs/',
                                                            ]]
collected_outputs_directory = find_first_extant_path(known_collected_outputs_paths)
assert collected_outputs_directory.exists(), f"collected_outputs_directory: {collected_outputs_directory} does not exist! Is the right computer's config commented out above?"
print(f'collected_outputs_directory: {collected_outputs_directory}')

known_bad_sessions: List[IdentifyingContext] = UserAnnotationsManager.get_hardcoded_bad_sessions()
known_bad_session_strs = [str(v.get_description()) for v in known_bad_sessions]


## Index exported H5 files


In [ ]:
cuttoff_date = datetime(2026, 11, 2)
print(f'cuttoff_date: {cuttoff_date}')

parsed_h5_files_df, h5_contexts_paths_dict = load_across_sessions_exported_h5_files(collected_outputs_directory=collected_outputs_directory, cuttoff_date=cuttoff_date,
                                                                                    known_bad_session_strs=known_bad_session_strs, debug_print=True)
parsed_h5_files_df['file_type'].unique()
parsed_h5_files_df


## Filter to `decoded_posteriors` (archive parity)


In [ ]:
## INPUTS: parsed_h5_files_df
decoded_posteriors_parsed_h5_files_df = parsed_h5_files_df[parsed_h5_files_df['file_type'] == 'decoded_posteriors']

print(decoded_posteriors_parsed_h5_files_df['custom_replay_name'].unique())
# matching_custom_replay_name_str: str = "withNormalComputedReplays-frateThresh_5.0-qclu_[1, 2]"
# matching_custom_replay_name_str: str = "withNormalComputedReplays-frateThresh_5.0-qclu_[1, 2, 4, 6, 7, 9]"
# matching_custom_replay_name_str: str = 'withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 9]-frateThresh_5.0'
# matching_custom_replay_name_str: str = 'withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 8, 9]-frateThresh_2.0'
matching_custom_replay_name_str: str = 'withNormalComputedReplays-qclu-[1, 2, 4, 6, 7, 8, 9]-frateThresh-2.0'
# compare_custom_replay_name_col_name: str = 'custom_replay_name'
compare_custom_replay_name_col_name: str = '_comparable_custom_replay_name'

decoding_time_bin_size_str: str = '0.025'
# decoding_time_bin_size_str: str = '0.075'

assert compare_custom_replay_name_col_name in decoded_posteriors_parsed_h5_files_df.columns
decoded_posteriors_parsed_h5_files_df = decoded_posteriors_parsed_h5_files_df.pho.constrain_df_cols(decoding_time_bin_size_str=decoding_time_bin_size_str, **{compare_custom_replay_name_col_name:matching_custom_replay_name_str}, should_drop_constrained_columns=False)
decoded_posteriors_parsed_h5_files_df = decoded_posteriors_parsed_h5_files_df.sort_values(['session', 'file_type', compare_custom_replay_name_col_name, 'decoding_time_bin_size_str', 'export_datetime']).reset_index(drop=True)
decoded_posteriors_parsed_h5_files_df = get_only_most_recent_session_files(parsed_paths_df=deepcopy(decoded_posteriors_parsed_h5_files_df), group_column_names=['session', compare_custom_replay_name_col_name, 'file_type', 'decoding_time_bin_size_str'])
decoded_posteriors_h5_files = [Path(v.as_posix()).resolve() for v in decoded_posteriors_parsed_h5_files_df['path'].to_list()]
decoded_posteriors_parsed_h5_files_df
## OUTPUTS: decoded_posteriors_h5_files, decoded_posteriors_parsed_h5_files_df


In [ ]:
decoded_posteriors_h5_files


## Array load (existing `LoadedPosteriorContainer` path)


In [ ]:
all_sessions_exported_posteriors_dict, all_sessions_exported_posteriors_data_only_dict = LoadedPosteriorContainer.load_batch_hdf5_exports(exported_posterior_data_h5_files=decoded_posteriors_h5_files)
## OUTPUTS: all_sessions_exported_posteriors_dict, all_sessions_exported_posteriors_data_only_dict
list(all_sessions_exported_posteriors_dict.keys())


## Recover original `SingleEpochDecodedResult` objects via `read_hdf`

Batch `(decoded_posteriors).h5` layout is per-epoch under `…/save_decoded_posteriors_to_HDF5/{decoder}/{laps|ripple}/{epoch_idx}/` — not the `DecodedFilterEpochsResult.read_hdf` `{key}/epochs/{i}` layout.


## Reassemble approximate `DecodedFilterEpochsResult` containers

Batch PosteriorExporting files do not use `DecodedFilterEpochsResult.to_hdf` layout, so wrap each recovered `SingleEpochDecodedResult` via `init_from_single_epoch_result` and merge with `init_by_merging_multiple_epoch_results`. `decoding_time_bin_size` comes from the filename `_tbin-…` (container-level attrs / full `filter_epochs` / `pos_bin_edges` may be incomplete vs the original export).

In [ ]:
## Recover from the first filtered file (extend to all paths as needed):
assert len(decoded_posteriors_h5_files) > 2, "no decoded_posteriors h5 files after filtering"
active_posterior_h5_path: Path = decoded_posteriors_h5_files[1]
print(f'active_posterior_h5_path: {active_posterior_h5_path}')

In [ ]:
from neuropy.utils.result_context import IdentifyingContext
from pyphoplacecellanalysis.Pho2D.stacked_epoch_slices import PhoPaginatedMultiDecoderDecodedEpochsWindow, DecodedEpochSlicesPaginatedFigureController
from pyphoplacecellanalysis.SpecificResults.PhoDiba2023Paper import HDF5RecoveredPosteriorRippleViewer

# (app, window, pagination_controller_dict, decoder_ripple_dict, scores_df, shuffle_df), overlays = HDF5RecoveredPosteriorRippleViewer.plot_session(
#     active_posterior_h5_path,
#     collected_outputs_directory=collected_outputs_directory,
#     decoded_posteriors_parsed_h5_files_df=decoded_posteriors_parsed_h5_files_df,
#     matching_custom_replay_name_str=matching_custom_replay_name_str,
#     decoding_time_bin_size_str=decoding_time_bin_size_str,
#     cuttoff_date=cuttoff_date,
#     overlay_included_columns=['wcorr', 'wcorr_z', 'mseq_tcov', 'mseq_tcov_z'], # , 'best_overall_mseq_tcov_z'
# )


## OUTPUTS: scores_df, shuffle_df, decoder_ripple_dict (filter_epochs enriched with score + shuffle z cols)
viewer: HDF5RecoveredPosteriorRippleViewer = HDF5RecoveredPosteriorRippleViewer.init_from_active_posterior_h5_path(
    active_posterior_h5_path,
    collected_outputs_directory=collected_outputs_directory,
    decoded_posteriors_parsed_h5_files_df=decoded_posteriors_parsed_h5_files_df,
    matching_custom_replay_name_str=matching_custom_replay_name_str,
    decoding_time_bin_size_str=decoding_time_bin_size_str,
    cuttoff_date=cuttoff_date,
    # overlay_included_columns=['wcorr', 'wcorr_z', 'mseq_tcov', 'mseq_tcov_p',
    # 'mseq_len_ignoring_intrusions_and_repeats_p',
    # 'spearman_p'], # , 'best_overall_mseq_tcov_z'
    sort_by='best_overall_wcorr_z', # None keeps temporal order; ascending=False => strongest first
    # sort_by='best_overall_wcorr_z', # None keeps temporal order; ascending=False => strongest first
    sort_ascending=False,
)
app, window, pagination_controller_dict = viewer.app, viewer.window, viewer.pagination_controller_dict
decoder_ripple_dict, scores_df, shuffle_df, overlays = viewer.decoder_ripple_dict, viewer.scores_df, viewer.shuffle_df, viewer.overlays
## OUTPUTS: viewer, scores_df, shuffle_df, decoder_ripple_dict (filter_epochs enriched with score + shuffle z cols)

In [ ]:
shuffle_df_columns: List[str] = list(shuffle_df.columns)
# [k for k in shuffle_df_columns if ('_z' in k) ]
[k for k in shuffle_df_columns if ('_p' in k) ]


In [ ]:
# print(list(scores_df.columns))

# identity / epoch
['start', 'stop', 'label', 'duration', 'n_unique_aclus',

# direction / best-dir indices
'P_LR', 'is_most_likely_direction_LR',
'long_best_dir_decoder_IDX', 'short_best_dir_decoder_IDX',

# raw wcorr (4 decoders)
'wcorr_long_LR', 'wcorr_long_RL', 'wcorr_short_LR', 'wcorr_short_RL',

# spearman raw
'spearman_long_LR', 'spearman_long_RL', 'spearman_short_LR', 'spearman_short_RL',

# mseq length heuristics (raw)
'mseq_len_long_LR', 'mseq_len_long_RL', 'mseq_len_short_LR', 'mseq_len_short_RL',
'mseq_len_ignoring_intrusions_long_LR', 'mseq_len_ignoring_intrusions_long_RL', 'mseq_len_ignoring_intrusions_short_LR', 'mseq_len_ignoring_intrusions_short_RL',
'mseq_len_ignoring_intrusions_and_repeats_long_LR', 'mseq_len_ignoring_intrusions_and_repeats_long_RL', 'mseq_len_ignoring_intrusions_and_repeats_short_LR', 'mseq_len_ignoring_intrusions_and_repeats_short_RL',
'mseq_len_ratio_ignoring_intrusions_and_repeats_long_LR', 'mseq_len_ratio_ignoring_intrusions_and_repeats_long_RL', 'mseq_len_ratio_ignoring_intrusions_and_repeats_short_LR', 'mseq_len_ratio_ignoring_intrusions_and_repeats_short_RL',

# mseq coverage / travel / forwardicity (raw)
'mseq_tcov_long_LR', 'mseq_tcov_long_RL', 'mseq_tcov_short_LR', 'mseq_tcov_short_RL',
'mseq_dtrav_long_LR', 'mseq_dtrav_long_RL', 'mseq_dtrav_short_LR', 'mseq_dtrav_short_RL',
'forwardicity_long_LR', 'forwardicity_long_RL', 'forwardicity_short_LR', 'forwardicity_short_RL',

# alignment ratios (raw)
'ratio_major_aligned_bins_long_LR', 'ratio_major_aligned_bins_long_RL', 'ratio_major_aligned_bins_short_LR', 'ratio_major_aligned_bins_short_RL',
'ratio_decoder_aligned_bins_long_LR', 'ratio_decoder_aligned_bins_long_RL', 'ratio_decoder_aligned_bins_short_LR', 'ratio_decoder_aligned_bins_short_RL',

# jump / travel / coverage (raw)
'avg_jump_cm_long_LR', 'avg_jump_cm_long_RL', 'avg_jump_cm_short_LR', 'avg_jump_cm_short_RL',
'travel_long_LR', 'travel_long_RL', 'travel_short_LR', 'travel_short_RL',
'coverage_long_LR', 'coverage_long_RL', 'coverage_short_LR', 'coverage_short_RL',

# BEGIN SHUFFLES _____________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________ #
## wcorr shuffle:
'wcorr_z_long', 'wcorr_z_short',

# spearman shuffle (z / p / n_extreme)
'spearman_z_long_LR', 'spearman_z_long_RL', 'spearman_z_short_LR', 'spearman_z_short_RL',
'spearman_p_long_LR', 'spearman_p_long_RL', 'spearman_p_short_LR', 'spearman_p_short_RL',
'spearman_n_extreme_long_LR', 'spearman_n_extreme_long_RL', 'spearman_n_extreme_short_LR', 'spearman_n_extreme_short_RL',

# mseq_len shuffle
'mseq_len_z_long_LR', 'mseq_len_z_long_RL', 'mseq_len_z_short_LR', 'mseq_len_z_short_RL',
'mseq_len_p_long_LR', 'mseq_len_p_long_RL', 'mseq_len_p_short_LR', 'mseq_len_p_short_RL',
'mseq_len_n_extreme_long_LR', 'mseq_len_n_extreme_long_RL', 'mseq_len_n_extreme_short_LR', 'mseq_len_n_extreme_short_RL',

# mseq_len_ignoring_intrusions shuffle
'mseq_len_ignoring_intrusions_z_long_LR', 'mseq_len_ignoring_intrusions_z_long_RL', 'mseq_len_ignoring_intrusions_z_short_LR', 'mseq_len_ignoring_intrusions_z_short_RL',
'mseq_len_ignoring_intrusions_p_long_LR', 'mseq_len_ignoring_intrusions_p_long_RL', 'mseq_len_ignoring_intrusions_p_short_LR', 'mseq_len_ignoring_intrusions_p_short_RL',
'mseq_len_ignoring_intrusions_n_extreme_long_LR', 'mseq_len_ignoring_intrusions_n_extreme_long_RL', 'mseq_len_ignoring_intrusions_n_extreme_short_LR', 'mseq_len_ignoring_intrusions_n_extreme_short_RL',

# mseq_len_ignoring_intrusions_and_repeats shuffle
'mseq_len_ignoring_intrusions_and_repeats_z_long_LR', 'mseq_len_ignoring_intrusions_and_repeats_z_long_RL', 'mseq_len_ignoring_intrusions_and_repeats_z_short_LR', 'mseq_len_ignoring_intrusions_and_repeats_z_short_RL',
'mseq_len_ignoring_intrusions_and_repeats_p_long_LR', 'mseq_len_ignoring_intrusions_and_repeats_p_long_RL', 'mseq_len_ignoring_intrusions_and_repeats_p_short_LR', 'mseq_len_ignoring_intrusions_and_repeats_p_short_RL',
'mseq_len_ignoring_intrusions_and_repeats_n_extreme_long_LR', 'mseq_len_ignoring_intrusions_and_repeats_n_extreme_long_RL', 'mseq_len_ignoring_intrusions_and_repeats_n_extreme_short_LR', 'mseq_len_ignoring_intrusions_and_repeats_n_extreme_short_RL',

# mseq_len_ratio_ignoring_intrusions_and_repeats shuffle
'mseq_len_ratio_ignoring_intrusions_and_repeats_z_long_LR', 'mseq_len_ratio_ignoring_intrusions_and_repeats_z_long_RL', 'mseq_len_ratio_ignoring_intrusions_and_repeats_z_short_LR', 'mseq_len_ratio_ignoring_intrusions_and_repeats_z_short_RL',
'mseq_len_ratio_ignoring_intrusions_and_repeats_p_long_LR', 'mseq_len_ratio_ignoring_intrusions_and_repeats_p_long_RL', 'mseq_len_ratio_ignoring_intrusions_and_repeats_p_short_LR', 'mseq_len_ratio_ignoring_intrusions_and_repeats_p_short_RL',
'mseq_len_ratio_ignoring_intrusions_and_repeats_n_extreme_long_LR', 'mseq_len_ratio_ignoring_intrusions_and_repeats_n_extreme_long_RL', 'mseq_len_ratio_ignoring_intrusions_and_repeats_n_extreme_short_LR', 'mseq_len_ratio_ignoring_intrusions_and_repeats_n_extreme_short_RL',

# mseq_tcov / mseq_dtrav / forwardicity shuffle
'mseq_tcov_z_long_LR', 'mseq_tcov_z_long_RL', 'mseq_tcov_z_short_LR', 'mseq_tcov_z_short_RL',
'mseq_tcov_p_long_LR', 'mseq_tcov_p_long_RL', 'mseq_tcov_p_short_LR', 'mseq_tcov_p_short_RL',
'mseq_tcov_n_extreme_long_LR', 'mseq_tcov_n_extreme_long_RL', 'mseq_tcov_n_extreme_short_LR', 'mseq_tcov_n_extreme_short_RL',
'mseq_dtrav_z_long_LR', 'mseq_dtrav_z_long_RL', 'mseq_dtrav_z_short_LR', 'mseq_dtrav_z_short_RL',
'mseq_dtrav_p_long_LR', 'mseq_dtrav_p_long_RL', 'mseq_dtrav_p_short_LR', 'mseq_dtrav_p_short_RL',
'mseq_dtrav_n_extreme_long_LR', 'mseq_dtrav_n_extreme_long_RL', 'mseq_dtrav_n_extreme_short_LR', 'mseq_dtrav_n_extreme_short_RL',
'forwardicity_z_long_LR', 'forwardicity_z_long_RL', 'forwardicity_z_short_LR', 'forwardicity_z_short_RL',
'forwardicity_p_long_LR', 'forwardicity_p_long_RL', 'forwardicity_p_short_LR', 'forwardicity_p_short_RL',
'forwardicity_n_extreme_long_LR', 'forwardicity_n_extreme_long_RL', 'forwardicity_n_extreme_short_LR', 'forwardicity_n_extreme_short_RL',

# alignment-ratio shuffle
'ratio_major_aligned_bins_z_long_LR', 'ratio_major_aligned_bins_z_long_RL', 'ratio_major_aligned_bins_z_short_LR', 'ratio_major_aligned_bins_z_short_RL',
'ratio_major_aligned_bins_p_long_LR', 'ratio_major_aligned_bins_p_long_RL', 'ratio_major_aligned_bins_p_short_LR', 'ratio_major_aligned_bins_p_short_RL',
'ratio_major_aligned_bins_n_extreme_long_LR', 'ratio_major_aligned_bins_n_extreme_long_RL', 'ratio_major_aligned_bins_n_extreme_short_LR', 'ratio_major_aligned_bins_n_extreme_short_RL',
'ratio_decoder_aligned_bins_z_long_LR', 'ratio_decoder_aligned_bins_z_long_RL', 'ratio_decoder_aligned_bins_z_short_LR', 'ratio_decoder_aligned_bins_z_short_RL',
'ratio_decoder_aligned_bins_p_long_LR', 'ratio_decoder_aligned_bins_p_long_RL', 'ratio_decoder_aligned_bins_p_short_LR', 'ratio_decoder_aligned_bins_p_short_RL',
'ratio_decoder_aligned_bins_n_extreme_long_LR', 'ratio_decoder_aligned_bins_n_extreme_long_RL', 'ratio_decoder_aligned_bins_n_extreme_short_LR', 'ratio_decoder_aligned_bins_n_extreme_short_RL',

# jump / travel / coverage shuffle
'avg_jump_cm_z_long_LR', 'avg_jump_cm_z_long_RL', 'avg_jump_cm_z_short_LR', 'avg_jump_cm_z_short_RL',
'avg_jump_cm_p_long_LR', 'avg_jump_cm_p_long_RL', 'avg_jump_cm_p_short_LR', 'avg_jump_cm_p_short_RL',
'avg_jump_cm_n_extreme_long_LR', 'avg_jump_cm_n_extreme_long_RL', 'avg_jump_cm_n_extreme_short_LR', 'avg_jump_cm_n_extreme_short_RL',
'travel_z_long_LR', 'travel_z_long_RL', 'travel_z_short_LR', 'travel_z_short_RL',
'travel_p_long_LR', 'travel_p_long_RL', 'travel_p_short_LR', 'travel_p_short_RL',
'travel_n_extreme_long_LR', 'travel_n_extreme_long_RL', 'travel_n_extreme_short_LR', 'travel_n_extreme_short_RL',
'coverage_z_long_LR', 'coverage_z_long_RL', 'coverage_z_short_LR', 'coverage_z_short_RL',
'coverage_p_long_LR', 'coverage_p_long_RL', 'coverage_p_short_LR', 'coverage_p_short_RL',
'coverage_n_extreme_long_LR', 'coverage_n_extreme_long_RL', 'coverage_n_extreme_short_LR', 'coverage_n_extreme_short_RL',

# session / export meta
'ripple_start_t', 'export_date', 'session_name', 'time_bin_size',
'delta_aligned_start_t', 'pre_post_delta_category',
'is_user_annotated_epoch', 'is_valid_epoch',

# derived best_overall_*_z
'best_overall_wcorr_z', 'best_overall_spearman_z', 'best_overall_coverage_z',
'best_overall_mseq_len_z', 'best_overall_mseq_len_ignoring_intrusions_z',
'best_overall_mseq_len_ignoring_intrusions_and_repeats_z',
'best_overall_mseq_len_ratio_ignoring_intrusions_and_repeats_z',
'best_overall_mseq_tcov_z', 'best_overall_mseq_dtrav_z', 'best_overall_forwardicity_z',
'best_overall_ratio_major_aligned_bins_z', 'best_overall_ratio_decoder_aligned_bins_z',
'best_overall_avg_jump_cm_z', 'best_overall_travel_z',
]

In [ ]:
viewer.window.add_data_overlays(included_columns=['wcorr', 'wcorr_z', 'mseq_tcov', 'mseq_tcov_p',
    'mseq_len_ignoring_intrusions_and_repeats_p',
    'spearman_p'])

In [ ]:
## Debug NaN overlay z: raw CSVs + merged filter_epochs
## INPUTS: scores_df, shuffle_df, decoder_ripple_dict
from neuropy.core.epoch import ensure_dataframe

debug_decoder_name: str = list(decoder_ripple_dict.keys())[0]
merged_epochs_df = ensure_dataframe(decoder_ripple_dict[debug_decoder_name].filter_epochs)
z_debug_cols = [c for c in ['start', 'wcorr', 'wcorr_z', 'best_overall_wcorr_z', 'mseq_tcov', 'mseq_tcov_z', 'best_overall_mseq_tcov_z', 'forwardicity', 'forwardicity_z', 'best_overall_forwardicity_z'] if c in merged_epochs_df.columns]
print(f'debug_decoder_name={debug_decoder_name!r}; scores_df={None if scores_df is None else scores_df.shape}; shuffle_df={None if shuffle_df is None else shuffle_df.shape}')
print(f'nan wcorr_z rows: {int(merged_epochs_df["wcorr_z"].isna().sum()) if "wcorr_z" in merged_epochs_df.columns else "missing"} / {len(merged_epochs_df)}')
display(merged_epochs_df[z_debug_cols])
display(scores_df)
display(shuffle_df)

In [ ]:
# 'spearman_long_LR', 
search_str: str = 'spearman'
print([k for k in scores_df.columns if (search_str in str(k))]) # ['spearman_long_LR', 'spearman_long_RL', 'spearman_short_LR', 'spearman_short_RL', 'spearman_z_long_LR', 'spearman_z_long_RL', 'spearman_z_short_LR', 'spearman_z_short_RL', 'spearman_p_long_LR', 'spearman_p_long_RL', 'spearman_p_short_LR', 'spearman_p_short_RL', 'spearman_n_extreme_long_LR', 'spearman_n_extreme_long_RL', 'spearman_n_extreme_short_LR', 'spearman_n_extreme_short_RL', 'best_overall_spearman_z']
print([k for k in shuffle_df.columns if (search_str in str(k))]) # ['spearman_long_LR', 'spearman_long_RL', 'spearman_short_LR', 'spearman_short_RL', 'spearman_z_long_LR', 'spearman_z_long_RL', 'spearman_z_short_LR', 'spearman_z_short_RL', 'spearman_p_long_LR', 'spearman_p_long_RL', 'spearman_p_short_LR', 'spearman_p_short_RL', 'spearman_n_extreme_long_LR', 'spearman_n_extreme_long_RL', 'spearman_n_extreme_short_LR', 'spearman_n_extreme_short_RL', 'best_overall_spearman_z']


# 2026-11-11 - Position-like bins

## Plot ripples in `PhoPaginatedMultiDecoderDecodedEpochsWindow`

Pipeline/`track_templates` are unavailable from HDF5 alone, so build each decoder's `DecodedEpochSlicesPaginatedFigureController` from the reassembled ripple `DecodedFilterEpochsResult`s (synthetic `xbin`, skip measured positions), then combine with `init_from_pagination_controller_dict`.

# 2026-11-04 - Test `PositionLikePosteriorScoring` for 1D Posteriors

In [ ]:
from pathlib import Path
import numpy as np
from pyphoplacecellanalysis.Pho2D.data_exporting import PosteriorExporting
from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import PositionLikePosteriorScoring

## INPUTS: active_posterior_h5_path
load_path = active_posterior_h5_path #Path(r"K:/scratch/collected_outputs/2026-10-28_GL-kdiba_gor01_two_2006-6-07_16-40-19__withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 8, 9]-frateThresh_2.0-(decoded_posteriors)_tbin-0.05.h5")
Assert.path_exists(load_path)
out_dict, _meta = PosteriorExporting.load_decoded_posteriors_from_HDF5(load_path=load_path, debug_print=False)



In [ ]:
from pyphocorehelpers.print_helpers import print_keys_if_possible

print_keys_if_possible('out_dict', out_dict, max_depth=2)



In [ ]:

print_keys_if_possible(f'out_dict[{decoder_name}][{epochs_name}]', out_dict[decoder_name][epochs_name], max_depth=1)

# 'p_x_given_n_full', 'p_x_given_n', 'p_x_given_n_grey'

# out_dict[decoder_name][epochs_name]
# out_dict[decoder_name][epochs_name]

In [ ]:

epochs_name = "laps"  # or "laps"
# epochs_name = "ripple"  # or "laps"
decoder_name = "long_LR"

# xbin = np.array([59.08, 61.9547, 64.8295, 67.7042, 70.579, 73.4537, 76.3285, 79.2032, 82.078, 84.9527, 87.8275, 90.7022, 93.5769, 96.4517, 99.3264, 102.201, 105.076, 107.951, 110.825, 113.7, 116.575, 119.45, 122.324, 125.199, 128.074, 130.949, 133.823, 136.698, 139.573, 142.448, 145.322, 148.197, 151.072, 153.947, 156.821, 159.696, 162.571, 165.446, 168.32, 171.195, 174.07, 176.945, 179.819, 182.694, 185.569, 188.444, 191.318, 194.193, 197.068, 199.943, 202.817, 205.692, 208.567, 211.442, 214.316, 217.191, 220.066, 222.941, 225.815, 228.69]) #TODO 2026-11-04 12:33: - [ ] KDiba Linear Track Session
n_pos, n_epoch_t = np.shape(out_dict[decoder_name][epochs_name]["p_x_given_n"][0]) ## uses the first epoch ([0])
print(f'n_pos: {n_pos}, n_epoch_t: {n_epoch_t}')
xbin = np.linspace(59.08, 228.69, (n_pos + 1))
xbin
print(f'len(xbin): {len(xbin)}, np.ptp(xbin): {np.ptp(xbin)}')
# Sanity check:
assert len(xbin) == (p_x_given_n.shape[0] + 1)


## INPUTS: out_dict, xbin, decoder_name, epochs_name
epoch_idx = 0
p_x_given_n = out_dict[decoder_name][epochs_name]["p_x_given_n"][epoch_idx]  # (n_pos, n_t)
# Primary API auto-dispatches to 1D when y_edges is omitted:
scoring_df = PositionLikePosteriorScoring.compute_and_plot_posterior_stack(p_x_given_n, x_edges=xbin, should_plot_results=True) ## plot a single one
# OUTPUTS: scoring_df, columns: t_bin_idx, score, sharpness, locality, rms_cm, is_position_like_1d, is_position_like
scoring_df
laps_detail_df

In [ ]:
epochs_name = "laps"  # or "laps"
# epochs_name = "ripple"  # or "laps"

decoder_name = "long_LR"

# xbin = np.array([59.08, 61.9547, 64.8295, 67.7042, 70.579, 73.4537, 76.3285, 79.2032, 82.078, 84.9527, 87.8275, 90.7022, 93.5769, 96.4517, 99.3264, 102.201, 105.076, 107.951, 110.825, 113.7, 116.575, 119.45, 122.324, 125.199, 128.074, 130.949, 133.823, 136.698, 139.573, 142.448, 145.322, 148.197, 151.072, 153.947, 156.821, 159.696, 162.571, 165.446, 168.32, 171.195, 174.07, 176.945, 179.819, 182.694, 185.569, 188.444, 191.318, 194.193, 197.068, 199.943, 202.817, 205.692, 208.567, 211.442, 214.316, 217.191, 220.066, 222.941, 225.815, 228.69]) #TODO 2026-11-04 12:33: - [ ] KDiba Linear Track Session
n_pos, n_epoch_t = np.shape(out_dict[decoder_name][epochs_name]["p_x_given_n"][0]) ## uses the first epoch ([0])
print(f'n_pos: {n_pos}, n_epoch_t: {n_epoch_t}')
xbin = np.linspace(59.08, 228.69, (n_pos + 1))
xbin
print(f'len(xbin): {len(xbin)}, np.ptp(xbin): {np.ptp(xbin)}')
# Sanity check:
# assert len(xbin) == (p_x_given_n.shape[0] + 1)
## OUTPUTS: xbin

In [ ]:
## OUTPUTS: epochs_name, decoder_name, xbin
# Compute them all at once for the loaded posteriors in `out_dict`: __________________________________________________________________________________________________________________________________________________________________________________________________________________ #
# Score all epochs for one decoder/epoch-type in the loaded H5 dict:
all_epochs_df, per_epoch_dfs = PositionLikePosteriorScoring.score_h5_loaded_decoder_epochs_1d(out_dict, decoder_name=decoder_name, epochs_name=epochs_name, x_edges=xbin)

## OUTPUTS: per_epoch_dfs
all_epochs_df


In [ ]:
per_epoch_dfs

In [ ]:
out_dict[decoder_name][epochs_name]["most_likely_position_indicies"][0]
# np.hstack(out_dict[decoder_name][epochs_name]["p_x_given_n"]['most_likely_position_indicies'])


In [ ]:
print_keys_if_possible('out_dict', out_dict, max_depth=1)

In [ ]:
print_keys_if_possible('out_dict[decoder_name][epochs_name]', out_dict[decoder_name][epochs_name], max_depth=1)

In [ ]:
out_dict

In [ ]:
from pathlib import Path
import numpy as np
from pyphoplacecellanalysis.Pho2D.data_exporting import PosteriorExporting
from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import PositionLikePosteriorScoring
from pyphoplacecellanalysis.SpecificResults.AcrossSessionResults import get_only_most_recent_session_files

## INPUTS: parsed_h5_files_df
decoded_posteriors_parsed_h5_files_df = parsed_h5_files_df[parsed_h5_files_df['file_type'] == 'decoded_posteriors']
matching_custom_replay_name_str: str = 'withNormalComputedReplays-qclu-[1, 2, 4, 6, 7, 8, 9]-frateThresh-2.0'
compare_custom_replay_name_col_name: str = '_comparable_custom_replay_name'

decoding_time_bin_size_str: str = '0.025'
# decoding_time_bin_size_str: str = '0.075'

assert compare_custom_replay_name_col_name in decoded_posteriors_parsed_h5_files_df.columns
decoded_posteriors_parsed_h5_files_df = decoded_posteriors_parsed_h5_files_df.pho.constrain_df_cols(decoding_time_bin_size_str=decoding_time_bin_size_str, **{compare_custom_replay_name_col_name:matching_custom_replay_name_str}, should_drop_constrained_columns=False)
decoded_posteriors_parsed_h5_files_df = decoded_posteriors_parsed_h5_files_df.sort_values(['session', 'file_type', compare_custom_replay_name_col_name, 'decoding_time_bin_size_str', 'export_datetime']).reset_index(drop=True) # Sort by columns: 'session' (ascending), 'custom_replay_name' (ascending) and 3 other columns
# get_only_most_recent_session_files _________________________________________________________________________________ #
## This is where we drop all but the most recent:
decoded_posteriors_parsed_h5_files_df = get_only_most_recent_session_files(parsed_paths_df=deepcopy(decoded_posteriors_parsed_h5_files_df), group_column_names=['session', compare_custom_replay_name_col_name, 'file_type', 'decoding_time_bin_size_str']) ## `decoded_posteriors_parsed_h5_files_df` still has it
decoded_posteriors_parsed_h5_files_df
decoded_posteriors_h5_files = [Path(v.as_posix()).resolve() for v in decoded_posteriors_parsed_h5_files_df['path'].to_list()]
decoded_posteriors_h5_files
## OUTPUTS: decoded_posteriors_h5_files, decoded_posteriors_parsed_h5_files_df


n_pos: int = 59 
xbin = np.linspace(59.08, 228.69, (n_pos + 1))
xbin

decoder_name = "long_LR"
summary, laps_detail_df, ripples_detail_df = PositionLikePosteriorScoring.calibrate_1d_pli_cutoffs_from_h5_files(h5_paths=decoded_posteriors_h5_files, x_edges=xbin, decoder_name=decoder_name)
summary
laps_detail_df

In [ ]:
_csv_out_path = f"{TODAY_DAY_DATE}_posteriorPositionLikeAnalysis_laps_detail_df.csv"

_out_parent_path = across_session_inst_fr_path.parent.joinpath('OUTPUTS')
_out_parent_path.mkdir(exist_ok=True)
print(f'saving to _out_parent_path: "{_out_parent_path.as_posix()}"...')

_out_results_dict = {f"{TODAY_DAY_DATE}_posteriorPositionLikeAnalysis_laps_detail_df.csv": laps_detail_df,
    f"{TODAY_DAY_DATE}_posteriorPositionLikeAnalysis_ripples_detail_df.csv": ripples_detail_df,
    # f"{TODAY_DAY_DATE}_posteriorPositionLikeAnalysis_summary.csv": summary,
}
for k, df in _out_results_dict.items():
    _csv_out_path = _out_parent_path.joinpath(k)
    print(f'saving to "{_csv_out_path.as_posix()}"...')
    df.to_csv(_csv_out_path)



In [ ]:

laps_detail_df.to_csv(_csv_out_path)

In [ ]:
{'n_files': 14,
 'files': ['K:\\scratch\\collected_outputs\\2026-11-03_0830PM-kdiba_gor01_one_2006-6-08_14-26-15__withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 8, 9]-frateThresh_2.0-(decoded_posteriors)_tbin-0.025.h5',
  'K:\\scratch\\collected_outputs\\2026-11-03_0630PM-kdiba_gor01_one_2006-6-09_1-22-43__withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 8, 9]-frateThresh_2.0-(decoded_posteriors)_tbin-0.025.h5',
  'K:\\scratch\\collected_outputs\\2026-11-03_0430PM-kdiba_gor01_one_2006-6-12_15-55-31__withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 8, 9]-frateThresh_2.0-(decoded_posteriors)_tbin-0.025.h5',
  'K:\\scratch\\collected_outputs\\2026-11-03_0900PM-kdiba_gor01_two_2006-6-07_16-40-19__withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 8, 9]-frateThresh_2.0-(decoded_posteriors)_tbin-0.025.h5',
  'K:\\scratch\\collected_outputs\\2026-11-03_0500PM-kdiba_gor01_two_2006-6-08_21-16-25__withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 8, 9]-frateThresh_2.0-(decoded_posteriors)_tbin-0.025.h5',
  'K:\\scratch\\collected_outputs\\2026-11-03_1010PM-kdiba_gor01_two_2006-6-09_22-24-40__withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 8, 9]-frateThresh_2.0-(decoded_posteriors)_tbin-0.025.h5',
  'K:\\scratch\\collected_outputs\\2026-11-03_0420PM-kdiba_gor01_two_2006-6-12_16-53-46__withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 8, 9]-frateThresh_2.0-(decoded_posteriors)_tbin-0.025.h5',
  'K:\\scratch\\collected_outputs\\2026-11-03_0450PM-kdiba_pin01_one_11-02_17-46-44__withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 8, 9]-frateThresh_2.0-(decoded_posteriors)_tbin-0.025.h5',
  'K:\\scratch\\collected_outputs\\2026-11-03_0410PM-kdiba_pin01_one_11-03_12-3-25__withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 8, 9]-frateThresh_2.0-(decoded_posteriors)_tbin-0.025.h5',
  'K:\\scratch\\collected_outputs\\2026-11-03_0830PM-kdiba_pin01_one_fet11-01_12-58-54__withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 8, 9]-frateThresh_2.0-(decoded_posteriors)_tbin-0.025.h5',
  'K:\\scratch\\collected_outputs\\2026-11-04_0520AM-kdiba_vvp01_one_2006-4-09_17-29-30__withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 8, 9]-frateThresh_2.0-(decoded_posteriors)_tbin-0.025.h5',
  'K:\\scratch\\collected_outputs\\2026-11-04_0450AM-kdiba_vvp01_one_2006-4-10_12-25-50__withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 8, 9]-frateThresh_2.0-(decoded_posteriors)_tbin-0.025.h5',
  'K:\\scratch\\collected_outputs\\2026-11-03_0410PM-kdiba_vvp01_two_2006-4-09_16-40-54__withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 8, 9]-frateThresh_2.0-(decoded_posteriors)_tbin-0.025.h5',
  'K:\\scratch\\collected_outputs\\2026-11-03_0420PM-kdiba_vvp01_two_2006-4-10_12-58-3__withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 8, 9]-frateThresh_2.0-(decoded_posteriors)_tbin-0.025.h5'],
 'n_lap_bins': 234308,
 'n_ripple_bins': 41469,
 'lap_score_percentiles': {10: 0.04178734810165212,
  25: 0.05272383546619831,
  50: 0.26304133261152884,
  75: 0.5932671160832894,
  90: 0.7197276169786149,
  95: 0.7554207368816872},
 'ripple_score_percentiles': {10: 0.05745387436744643,
  25: 0.07112948221531606,
  50: 0.427796388472173,
  75: 0.6897689809198442,
  90: 0.7753890222387692,
  95: 0.8043207492499671},
 'suggested_position_like_score_cutoff_1d': 0.26304133261152884,
 'suggested_high_quality_score_cutoff_1d': 0.7197276169786149,
 'note': 'ripple_median > lap_median; using lap_median for position_like cutoff'}

In [ ]:
ERROR: could not parse curr_export_result_save_properties: ['kdiba', 'gor01', 'one', '2006-6-09_1-22-43', 'save_decoded_posteriors_to_HDF5'] for file: "K:/scratch/collected_outputs/2026-11-03_0630PM-kdiba_gor01_one_2006-6-09_1-22-43__withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 8, 9]-frateThresh_2.0-(decoded_posteriors)_tbin-0.025.h5". Skipping those properties but loading anyway.
session_key_str: "kdiba-gor01-one-2006-6-12_15-55-31"
session_key_parts: ['kdiba', 'gor01', 'one', '2006-6-12_15-55-31']
custom_replay_parts: []